# BCO7006 — Session 7
# Pandas Foundations: Finding Your Data
**Duration:** 30 min lecture

By the end of this notebook you will be able to:
1. Create and inspect a `Series` and a `DataFrame`
2. Read data from CSV with `pd.read_csv`
3. Inspect data with `.head()`, `.info()`, `.describe()`, `.dtypes`, `.shape`
4. **Select rows and columns with `.loc[]` and `.iloc[]`** — the most important skill in this session
5. Filter rows with boolean conditions
6. Sort data with `.sort_values()`

We will use a real-ish dataset: `sales_data.csv` (500 sales transactions).

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

# Tell pandas to show all columns when we print a DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("pandas version:", pd.__version__)

pandas version: 3.0.2


## 2. Series and DataFrame — the two core objects

A **Series** is a 1D labelled array. Think of it as one column.
A **DataFrame** is a 2D labelled table. Think of it as a spreadsheet.

In [2]:
# A Series — one column of data with an index
prices = pd.Series([10.50, 22.00, 8.75, 15.30], index=['apple', 'mango', 'banana', 'orange'])
print(prices)
print()
print("Type:", type(prices).__name__)
print("dtype:", prices.dtype)

apple     10.50
mango     22.00
banana     8.75
orange    15.30
dtype: float64

Type: Series
dtype: float64


In [3]:
# A DataFrame built from a dictionary — each key becomes a column
df_small = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Gadget X'],
    'price':   [29.99, 49.50, 199.00],
    'in_stock': [True, False, True]
})
df_small

,product,price,in_stock
0,Widget A,29.99,True
1,Widget B,49.50,False
2,Gadget X,199.00,True


## 3. Loading real data with `read_csv`

In practice you'll almost never type data by hand. You'll load it from a file.

In [4]:
df = pd.read_csv('sales_data.csv')
df.head()  # first 5 rows by default

,order_id,date,region,product,sales_rep,units,unit_price,customer_rating,revenue
0,1001,2024-01-01,West,Gadget Y,David,2,122.65,5.0,245.30
1,1002,2024-01-01,Central,Widget A,David,35,120.09,5.0,4203.15
2,1003,2024-01-01,East,Tool Z,David,42,135.90,2.0,5707.80
3,1004,2024-01-01,Central,Gadget X,Henry,34,28.10,3.0,955.40
4,1005,2024-01-02,Central,Gadget X,David,30,133.90,NaN,4017.00


## 4. Inspecting the data — always do this first

In [5]:
df.shape  # (rows, columns)

(500, 9)

In [6]:
df.info()  # column names, non-null counts, dtypes, memory

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         500 non-null    int64  
 1   date             500 non-null    str    
 2   region           500 non-null    str    
 3   product          500 non-null    str    
 4   sales_rep        500 non-null    str    
 5   units            500 non-null    int64  
 6   unit_price       500 non-null    float64
 7   customer_rating  435 non-null    float64
 8   revenue          500 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 35.3 KB


In [7]:
df.describe()  # summary statistics for numeric columns

,order_id,units,unit_price,customer_rating,revenue
count,500.000000,500.000000,500.000000,435.000000,500.000000
mean,1250.500000,25.288000,102.721220,3.774713,2566.782280
std,144.481833,13.938494,53.242713,1.238702,2073.108271
min,1001.000000,1.000000,10.060000,1.000000,53.910000
25%,1125.750000,13.000000,56.282500,3.000000,886.507500
50%,1250.500000,26.000000,102.565000,4.000000,1976.230000
75%,1375.250000,37.000000,146.530000,5.000000,3811.230000
max,1500.000000,49.000000,199.140000,5.000000,8964.940000


In [8]:
df.dtypes  # data type of each column

order_id             int64
date                   str
region                 str
product                str
sales_rep              str
units                int64
unit_price         float64
customer_rating    float64
revenue            float64
dtype: object

In [9]:
df.columns.tolist()  # just the column names

['order_id',
 'date',
 'region',
 'product',
 'sales_rep',
 'units',
 'unit_price',
 'customer_rating',
 'revenue']

## 5. Selecting columns

Three common ways:

In [10]:
# Single column — returns a Series
df['revenue'].head()

0     245.30
1    4203.15
2    5707.80
3     955.40
4    4017.00
Name: revenue, dtype: float64

In [11]:
# Multiple columns — returns a DataFrame. Note the DOUBLE brackets.
df[['region', 'product', 'revenue']].head()

,region,product,revenue
0,West,Gadget Y,245.30
1,Central,Widget A,4203.15
2,East,Tool Z,5707.80
3,Central,Gadget X,955.40
4,Central,Gadget X,4017.00


In [12]:
# Dot notation — works but DON'T rely on it (breaks if column has a space or matches a method name)
df.revenue.head()

0     245.30
1    4203.15
2    5707.80
3     955.40
4    4017.00
Name: revenue, dtype: float64

## 6. ⭐ `.loc[]` and `.iloc[]` — the most important skill in pandas

These let you select rows AND columns at the same time.

- **`.loc[row_labels, column_labels]`** — by LABEL (names, conditions)
- **`.iloc[row_positions, column_positions]`** — by POSITION (integers, like a list)

**Why both?** Sometimes you know the name, sometimes you know the position. Mixing them up is a classic bug source.

In [13]:
# .iloc — by position. Row 0, all columns
df.iloc[0]

order_id                 1001
date               2024-01-01
region                   West
product              Gadget Y
sales_rep               David
units                       2
unit_price             122.65
customer_rating           5.0
revenue                 245.3
Name: 0, dtype: object

In [14]:
# .iloc — first 3 rows, first 4 columns
df.iloc[0:3, 0:4]

,order_id,date,region,product
0,1001,2024-01-01,West,Gadget Y
1,1002,2024-01-01,Central,Widget A
2,1003,2024-01-01,East,Tool Z


In [15]:
# .iloc — last row, all columns
df.iloc[-1]

order_id                 1500
date               2024-05-04
region                  North
product              Gadget X
sales_rep               David
units                      47
unit_price             154.06
customer_rating           NaN
revenue               7240.82
Name: 499, dtype: object

In [16]:
# .loc — by label. Row with index 5, specific columns
df.loc[5, ['product', 'revenue']]

product    Widget A
revenue      901.03
Name: 5, dtype: object

In [17]:
# .loc — rows 0 to 4 (INCLUSIVE — different from .iloc!), specific columns
df.loc[0:4, ['region', 'product', 'revenue']]

,region,product,revenue
0,West,Gadget Y,245.30
1,Central,Widget A,4203.15
2,East,Tool Z,5707.80
3,Central,Gadget X,955.40
4,Central,Gadget X,4017.00


### ⚠ Gotcha: `.loc` slice is INCLUSIVE, `.iloc` slice is EXCLUSIVE

```python
df.loc[0:4]    # rows with labels 0, 1, 2, 3, 4  → 5 rows
df.iloc[0:4]   # rows at positions 0, 1, 2, 3    → 4 rows
```

Why? `.loc` uses labels, and label slicing in pandas is inclusive on both ends. `.iloc` follows Python list conventions.

In [18]:
# Demonstrate the difference
print("df.loc[0:4] has", len(df.loc[0:4]), "rows")
print("df.iloc[0:4] has", len(df.iloc[0:4]), "rows")

df.loc[0:4] has 5 rows
df.iloc[0:4] has 4 rows


## 7. Filtering rows with boolean conditions

In [19]:
# Single condition — sales over $5000
big_sales = df[df['revenue'] > 5000]
print(f"Found {len(big_sales)} sales over $5000")
big_sales.head()

Found 72 sales over $5000


,order_id,date,region,product,sales_rep,units,unit_price,customer_rating,revenue
2,1003,2024-01-01,East,Tool Z,David,42,135.90,2.0,5707.80
14,1015,2024-01-04,West,Gadget Y,Bob,49,177.90,3.0,8717.10
18,1019,2024-01-05,North,Tool Z,Eve,44,137.18,5.0,6035.92
28,1029,2024-01-08,West,Tool Z,Eve,49,111.37,1.0,5457.13
30,1031,2024-01-08,East,Widget B,Frank,45,144.02,5.0,6480.90


In [20]:
# Multiple conditions — use & (and), | (or), ~ (not). PARENTHESES are mandatory.
north_big = df[(df['region'] == 'North') & (df['revenue'] > 3000)]
print(f"North region with revenue > $3000: {len(north_big)} rows")
north_big.head()

North region with revenue > $3000: 42 rows


,order_id,date,region,product,sales_rep,units,unit_price,customer_rating,revenue
18,1019,2024-01-05,North,Tool Z,Eve,44,137.18,5.0,6035.92
70,1071,2024-01-18,North,Widget B,David,49,135.71,4.0,6649.79
76,1077,2024-01-20,North,Widget B,ALICE,23,174.28,5.0,4008.44
81,1082,2024-01-21,North,Tool Z,Alice,49,124.12,5.0,6081.88
124,1125,2024-02-01,North,Widget B,Henry,18,170.96,3.0,3077.28


In [21]:
# .isin() — match any value in a list
selected = df[df['product'].isin(['Widget A', 'Gadget X'])]
print(f"Widget A or Gadget X sales: {len(selected)} rows")
selected['product'].value_counts()

Widget A or Gadget X sales: 200 rows


product
Widget A    101
Gadget X     99
Name: count, dtype: int64

In [22]:
# Combining filters with .loc — best practice for filter + select columns together
df.loc[df['revenue'] > 5000, ['region', 'product', 'revenue']].head()

,region,product,revenue
2,East,Tool Z,5707.80
14,West,Gadget Y,8717.10
18,North,Tool Z,6035.92
28,West,Tool Z,5457.13
30,East,Widget B,6480.90


### ⚠ The `SettingWithCopyWarning`

When you do `df[df['x']>5]['y'] = 0`, pandas can't tell if you want to modify the original or a copy. Use `.loc` instead:

```python
# BAD — may warn, may not work
df[df['region'] == 'North']['revenue'] = 0

# GOOD — clear, works
df.loc[df['region'] == 'North', 'revenue'] = 0
```

## 8. Sorting with `.sort_values()`

In [23]:
# Sort by one column, descending
df.sort_values('revenue', ascending=False).head()

,order_id,date,region,product,sales_rep,units,unit_price,customer_rating,revenue
52,1053,2024-01-14,West,Widget B,Alice,46,194.89,5.0,8964.94
427,1428,2024-04-16,Central,Gadget Y,David,45,198.40,1.0,8928.00
403,1404,2024-04-10,Central,Gadget Y,Alice,47,186.76,NaN,8777.72
14,1015,2024-01-04,West,Gadget Y,Bob,49,177.90,3.0,8717.10
188,1189,2024-02-17,North,Widget A,Frank,49,171.72,5.0,8414.28


In [24]:
# Sort by multiple columns — region ascending, then revenue descending within each region
df.sort_values(['region', 'revenue'], ascending=[True, False]).head(10)

,order_id,date,region,product,sales_rep,units,unit_price,customer_rating,revenue
427,1428,2024-04-16,Central,Gadget Y,David,45,198.40,1.0,8928.00
403,1404,2024-04-10,Central,Gadget Y,Alice,47,186.76,NaN,8777.72
82,1083,2024-01-21,Central,Widget A,Alice,42,188.77,4.0,7928.34
494,1495,2024-05-03,Central,Tool Z,Bob,37,193.37,2.0,7154.69
296,1297,2024-03-15,Central,Tool Z,Grace,46,147.07,3.0,6765.22
191,1192,2024-02-17,Central,Tool Z,Frank,40,155.66,5.0,6226.40
139,1140,2024-02-04,Central,Tool Z,Carol,34,180.38,5.0,6132.92
126,1127,2024-02-01,Central,Gadget Y,Bob,37,164.62,5.0,6090.94
395,1396,2024-04-08,Central,Tool Z,David,31,191.68,NaN,5942.08
83,1084,2024-01-21,Central,Widget A,Eve,35,167.10,3.0,5848.50


In [25]:
# Sort doesn't modify in place by default — assign or use inplace=True
df_sorted = df.sort_values('date')
df_sorted.head()

,order_id,date,region,product,sales_rep,units,unit_price,customer_rating,revenue
0,1001,2024-01-01,West,Gadget Y,David,2,122.65,5.0,245.30
1,1002,2024-01-01,Central,Widget A,David,35,120.09,5.0,4203.15
2,1003,2024-01-01,East,Tool Z,David,42,135.90,2.0,5707.80
3,1004,2024-01-01,Central,Gadget X,Henry,34,28.10,3.0,955.40
4,1005,2024-01-02,Central,Gadget X,David,30,133.90,NaN,4017.00


## 9. Putting it together — answering a business question

> *"What were our 5 highest-revenue sales in the North region?"*

In [26]:
(df
 .loc[df['region'] == 'North', ['date', 'product', 'sales_rep', 'revenue']]
 .sort_values('revenue', ascending=False)
 .head(5)
)

,date,product,sales_rep,revenue
188,2024-02-17,Widget A,Frank,8414.28
323,2024-03-21,Gadget X,Bob,7563.84
439,2024-04-19,Widget A,Alice,7412.39
499,2024-05-04,Gadget X,David,7240.82
182,2024-02-15,Widget B,Grace,7102.68


This **method chaining** pattern — filter → select → sort → limit — is the workhorse of pandas analysis. Get comfortable reading it.

## 10. Summary

| Task | Method |
|---|---|
| Load CSV | `pd.read_csv('file.csv')` |
| First/last rows | `.head()`, `.tail()` |
| Shape, info | `.shape`, `.info()`, `.describe()`, `.dtypes` |
| Column by name | `df['col']` or `df[['col1', 'col2']]` |
| Row + col by **label** | `df.loc[rows, cols]` |
| Row + col by **position** | `df.iloc[rows, cols]` |
| Filter rows | `df[condition]` or `df.loc[condition, cols]` |
| Sort | `df.sort_values('col', ascending=False)` |

**Next:** in your pair programming activity, you'll apply all of this to answer real questions about the sales data.